# Clip Infrastructure Layers to Africa (ArcPy)

**Purpose.** Clip annual infrastructure shapefiles to the study-area boundary.

**Inputs.** Annual shapefiles and the Africa boundary shapefile specified in the notebook.

**Outputs.** Clipped annual infrastructure layers.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | ArcPy environment and annual clipping configuration
import arcpy
from arcpy import env
import os
import time
from datetime import datetime

YEAR = 2025

arcpy.env.overwriteOutput = True
arcpy.env.workspace = r"E:\Africa_Infrastructure_Data\infrastructure"

print("=" * 80)
print("=" * 80)


In [ ]:
# Cell | Boundary, input and output validation
boundary_file = r"E:\Africa_Infrastructure_Data\boundaries\Country_dissolve.shp"

input_dir = rf"E:\Africa_Infrastructure_Data\infrastructure\shp\{YEAR}"
output_dir = rf"E:\Africa_Infrastructure_Data\infrastructure\clip\{YEAR}"

MAX_RETRIES = 3
RETRY_DELAY = 2

shp_files = [f for f in os.listdir(input_dir) if f.lower().endswith('.shp')]


if not arcpy.Exists(boundary_file):
    raise FileNotFoundError(f"边界文件不存在: {boundary_file}")

if not os.path.exists(input_dir):
    raise FileNotFoundError(f"输入目录不存在: {input_dir}")

if shp_files:
    for i, f in enumerate(shp_files, 1):
        print(f"  {i}. {f}")
else:
    pass



In [ ]:
# Cell | Geometry repair and study-area clipping function
# Repair input geometries before clipping to the dissolved Africa boundary.
def clip_feature(input_file, boundary, output_file):
    """Process line and polygon features."""
    temp_repaired = None
    
    try:
        if not arcpy.Exists(input_file):
            return {'success': False, 'message': '输入文件不存在'}
        
        original_count = int(arcpy.GetCount_management(input_file)[0])
        
        if original_count == 0:
            return {'success': False, 'message': '输入文件为空', 'original_count': 0}
        
        output_dir = os.path.dirname(output_file)
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        temp_repaired = "in_memory\\temp_repaired"
        
        arcpy.RepairGeometry_management(input_file, "DELETE_NULL")
        
        arcpy.CopyFeatures_management(input_file, temp_repaired)
        
        arcpy.Clip_analysis(temp_repaired, boundary, output_file)
        
        clipped_count = int(arcpy.GetCount_management(output_file)[0])
        
        if arcpy.Exists(temp_repaired):
            arcpy.Delete_management(temp_repaired)
        
        if clipped_count > 0:
            return {
                'success': True,
                'original_count': original_count,
                'clipped_count': clipped_count
            }
        else:
            return {
                'success': False,
                'message': '裁剪后无数据',
                'original_count': original_count
            }
    
    except Exception as e:
        # Continue gracefully if this estimation step fails.
        try:
            if temp_repaired and arcpy.Exists(temp_repaired):
                arcpy.Delete_management(temp_repaired)
        except:
            pass
        return {'success': False, 'message': f'错误: {str(e)}'}



In [ ]:
# Cell | Batch-processing status ledger
# Track processed files, skipped layers, failures and elapsed time.
stats = {
    'total_files': 0,
    'successful': 0,
    'skipped': 0,
    'failed': 0,
    'total_time': 0,
    'details': []
}

print("=" * 80)
print("=" * 80)

In [ ]:
# Cell | Retriable clipping workflow for annual layers
overall_start = time.time()


if not os.path.exists(output_dir):
    try:
        os.makedirs(output_dir)
    except Exception as e:
        pass

# Apply clipping to every source Shapefile with retry handling.
for file_name in shp_files:
    input_file = os.path.join(input_dir, file_name)
    output_file = os.path.join(output_dir, file_name)
    
    stats['total_files'] += 1
    
    file_start = time.time()
    
    result = None
    retry_count = 0
    
    while retry_count <= MAX_RETRIES:
        result = clip_feature(input_file, boundary_file, output_file)
        
        if result['success']:
            break
        else:
            if '输入文件不存在' in result['message'] or '裁剪后无数据' in result['message'] or '输入文件为空' in result['message']:
                break
            
            retry_count += 1
            if retry_count <= MAX_RETRIES:
                time.sleep(RETRY_DELAY)
    
    file_time = time.time() - file_start
    
    if result['success']:
        stats['successful'] += 1
        stats['details'].append({
            'file': file_name,
            'status': 'success',
            'original': result['original_count'],
            'clipped': result['clipped_count'],
            'time': file_time
        })
    else:
        if 'original_count' in result:
            stats['skipped'] += 1
        else:
            stats['failed'] += 1

overall_time = time.time() - overall_start
stats['total_time'] = overall_time

In [ ]:
# Cell | Batch clipping summary
if stats['successful'] > 0:
    avg_time = sum(d['time'] for d in stats['details'] if d['status'] == 'success') / stats['successful']
    
    for detail in stats['details']:
        if detail['status'] == 'success':
            pass

print("\n" + "=" * 80)
print("=" * 80)
